# 🎓 Socratic Science Tutor — Dual T4 GPU Fine-Tuning

### ⚡ Master Features:
- **Dataset**: `/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl` (auto-synced to `Susu11/socratic_idea_expansion`)
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct`
- **Target Model Output**: [`Susu11/socratic_qwen8b`](https://huggingface.co/Susu11/socratic_qwen8b)
- **Per-Epoch Upload**: Automatically saves and pushes checkpoints to Hugging Face after **each epoch**!
- **GGUF Export**: Converts merged model to `.gguf` (for Ollama / llama.cpp / LM Studio) and uploads directly to Hugging Face!

---

## 1. Install Dependencies

In [23]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes wandb huggingface_hub gguf
print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


## 2. Verify Dual T4 GPUs

In [24]:
import torch

gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Total GPUs detected: {gpu_count}")
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  -> GPU {i}: {name} ({mem:.2f} GB VRAM)")

assert gpu_count >= 1, "⚠️ No GPU detected! Go to Notebook Settings -> Accelerator -> select GPU T4 x2."

CUDA Available: True
Total GPUs detected: 2
  -> GPU 0: Tesla T4 (15.64 GB VRAM)
  -> GPU 1: Tesla T4 (15.64 GB VRAM)


## 3. Automatic Authentication (HF_TOKEN & WANDB_TOKEN)

In [25]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

def clean(v):
    return v.strip().strip("'").strip('"').strip() if v else None

secrets = UserSecretsClient()

# 1. Read HF_TOKEN
hf_token = None
for label in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "huggingface", "HF"]:
    try:
        hf_token = clean(secrets.get_secret(label))
        if hf_token: break
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        login(token=hf_token, add_to_git_credential=True)
        print("✅ Hugging Face authentication successful!")
    except Exception as e:
        print(f"⚠️ HF login notice: {e}")
else:
    print("⚠️ HF_TOKEN not found in Kaggle Secrets.")

# 2. Read WANDB_TOKEN
wandb_token = None
for label in ["WANDB_TOKEN", "WANDB_API_KEY", "wandb_token", "wandb_key", "wandb"]:
    try:
        wandb_token = clean(secrets.get_secret(label))
        if wandb_token: break
    except Exception:
        pass

if wandb_token:
    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_PROJECT"] = "socratic-model-fine-tune"
    try:
        wandb.login(key=wandb_token)
        print("✅ Weights & Biases authentication successful!")
    except Exception as e:
        print(f"ℹ️ Wandb login notice: {e}")
        os.environ["WANDB_DISABLED"] = "true"
else:
    print("ℹ️ WANDB_TOKEN not found. Running in offline mode.")
    os.environ["WANDB_DISABLED"] = "true"

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
✅ Hugging Face authentication successful!
ℹ️ Wandb login notice: module 'wandb' has no attribute 'login'


## 4. Load Dataset from Kaggle Input & Sync to Hugging Face
Reads directly from `/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl`.

In [26]:
import os
import json
from datasets import Dataset, load_dataset

DATASET_REPO = "Susu11/socratic_idea_expansion"

candidate_paths = [
    "/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl",
    "/kaggle/input/socratic-final/socratic_dataset.jsonl",
    "socratic_dataset.jsonl"
]

target_file = None
for path in candidate_paths:
    if os.path.exists(path):
        target_file = path
        break

if target_file:
    print(f"📂 Found dataset file at: {target_file}")
    records = [json.loads(line) for line in open(target_file, "r", encoding="utf-8") if line.strip()]
    dataset = Dataset.from_list(records)
    print(f"✅ Successfully loaded {len(dataset)} conversation examples from disk!")
    
    if os.getenv("HF_TOKEN"):
        try:
            print(f"📤 Syncing to Hugging Face Hub: https://huggingface.co/datasets/{DATASET_REPO}...")
            dataset.push_to_hub(DATASET_REPO, private=False)
            print("🎉 Dataset synced to Hugging Face Hub successfully!")
        except Exception as e:
            print(f"ℹ️ Hub upload notice: {e}")
else:
    print(f"🌐 Local file not found, loading directly from Hub: {DATASET_REPO}...")
    dataset = load_dataset(DATASET_REPO, split="train")
    print(f"✅ Loaded {len(dataset)} examples from Hugging Face Hub!")

print("\n--- First Dialogue Sample ---")
for msg in dataset[0]["messages"]:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

📂 Found dataset file at: /kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl
✅ Successfully loaded 606 conversation examples from disk!
📤 Syncing to Hugging Face Hub: https://huggingface.co/datasets/Susu11/socratic_idea_expansion...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


🎉 Dataset synced to Hugging Face Hub successfully!

--- First Dialogue Sample ---
[SYSTEM]:
You are a Socratic Science Tutor for a Grade 10 student. Your goal is to guide the student to discover concepts through reasoning, NEVER by giving the final answer directly. Ask EXACTLY ONE question per turn. Keep responses to 1-3 sentences. If the student is stuck, provide a simpler analogy or break the concept into a smaller step. State your pedagogical goal inside <plan>...</plan> tags.

[USER]:
What is the difference between speed and velocity?

[ASSISTANT]:
<plan>Goal: Differentiate speed (scalar) from velocity (vector) using direction.</plan>
They are closely related, but there is one crucial difference. If a car drives in a perfect circle at a steady 50 km/h, is its speed changing?

[USER]:
No, the speedometer stays at 50 km/h.

[ASSISTANT]:
<plan>Goal: Introduce direction to define velocity.</plan>
Correct, the speed is constant. However, as the car goes around the circle, what is consta

## 5. Load Qwen2.5-7B-Instruct with 4-bit QLoRA across Dual T4 GPUs

In [27]:
import gc
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Clean previous VRAM allocations & release IPython traceback locks
for var in ["trainer", "model", "train_result"]:
    if var in globals():
        del globals()[var]
sys.last_traceback = None
sys.last_value = None
sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    print(f"GPU {i} Current VRAM in use: {alloc:.2f} GB")

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"\n🤖 Base Model: {BASE_MODEL_ID}")

# 2. Tokenizer and Chat Template
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
            for conv in batch["messages"]
        ]
    }

train_dataset = dataset.map(apply_chat_template, batched=True)
print("📋 Formatted chat text preview:")
print(train_dataset[0]["text"][:350] + "...\n")

# 3. QLoRA 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 4. Balanced Memory Allocation: Limit GPU 0 to 6GiB so model shards evenly across both GPUs
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    max_memory = {0: "6GiB", 1: "13GiB"}
else:
    max_memory = {0: "13GiB"}
print(f"Target GPU Memory Allocation: {max_memory}")

# 5. Load Model across GPUs
print("⏳ Loading model across both T4 GPUs (takes ~1-2 mins)... ")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 6. Prepare LoRA Model
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# Enable gradient checkpointing to save ~70% VRAM
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Mark model as parallel so Hugging Face Trainer never wraps it in torch.nn.DataParallel
setattr(model, "is_parallelizable", True)
setattr(model, "model_parallel", True)
if hasattr(model, "base_model") and hasattr(model.base_model, "model") and hasattr(model.base_model.model, "hf_device_map"):
    model.hf_device_map = model.base_model.model.hf_device_map

print("\n📊 Device Map:")
print(getattr(model, "hf_device_map", "auto"))
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

GPU 0 Current VRAM in use: 0.00 GB
GPU 1 Current VRAM in use: 0.00 GB

🤖 Base Model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


📊 Device Map:
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}

📊 Trainable parameters:
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 6. Train Socratic Model with SFTTrainer & W&B
- Since `model` is already a `PeftModel`, we pass `model=model` directly without duplicating `peft_config`.

In [28]:
import os
import inspect
import torch
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./socratic_tutor_lora"
HUB_MODEL_ID = "Susu11/socratic_qwen8b"

# Set model config dtype to float16
model.config.torch_dtype = torch.float16

# Base training arguments optimized for Dual T4 GPUs
# Note: fp16=False disables GradScaler, which avoids the BFloat16 unscale error on T4.
# The 4-bit base model still uses fast 16-bit tensor cores via bnb_4bit_compute_dtype=float16.
base_args = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "logging_steps": 5,
    "save_strategy": "epoch",
    "push_to_hub": bool(os.getenv("HF_TOKEN")),
    "hub_model_id": HUB_MODEL_ID if os.getenv("HF_TOKEN") else None,
    "hub_strategy": "every_save",
    "fp16": False,
    "bf16": False,
    "max_grad_norm": 0.3,
    "optim": "paged_adamw_8bit",
    "report_to": ["wandb"] if os.getenv("WANDB_API_KEY") else ["none"],
    "run_name": "kaggle-dual-t4-qwen-run",
}

# Filter parameters for SFTConfig
sft_config_params = inspect.signature(SFTConfig.__init__).parameters
if "loss_type" in sft_config_params:
    base_args["loss_type"] = "nll"

if "max_length" in sft_config_params:
    base_args["max_length"] = 2048
elif "max_seq_length" in sft_config_params:
    base_args["max_seq_length"] = 2048

if "dataset_text_field" in sft_config_params:
    base_args["dataset_text_field"] = "text"

valid_config_args = {k: v for k, v in base_args.items() if k in sft_config_params}
training_args = SFTConfig(**valid_config_args)

# Crucial: Prevent Trainer from wrapping model in torch.nn.DataParallel
training_args._n_gpu = 1

# Prevent TRL inspect signature crash
try:
    import trl.trainer.sft_trainer
    if hasattr(trl.trainer.sft_trainer, "_patch_chunked_ce_lm_head"):
        trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *a, **kw: None
except Exception:
    pass

# Configure SFTTrainer
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "args": training_args,
}

if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in valid_config_args:
    trainer_kwargs["dataset_text_field"] = "text"

if "max_seq_length" in sft_trainer_params and "max_seq_length" not in valid_config_args and "max_length" not in valid_config_args:
    trainer_kwargs["max_seq_length"] = 2048

trainer = SFTTrainer(**trainer_kwargs)
trainer.is_model_parallel = True

print("🚀 Starting Training on Dual T4 GPUs (syncing to HF at every epoch)... ")
train_result = trainer.train()
print("✅ Training Completed Successfully!")
print(f"📉 Final Training Loss: {train_result.training_loss:.4f}")

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
wandb.__spec__ is None

## 7. Live Test: Verify Socratic Persona & `<plan>` Tags

In [ ]:
import os
import glob
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

OUTPUT_DIR = "./socratic_tutor_lora"
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def get_best_adapter_dir(output_dir="./socratic_tutor_lora"):
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        return output_dir
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
        return checkpoints[-1]
    return output_dir

adapter_path = get_best_adapter_dir(OUTPUT_DIR)
print(f"📂 Loading trained adapter from: {adapter_path}...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

num_gpus = torch.cuda.device_count()
max_memory = {0: "6GiB", 1: "13GiB"} if num_gpus > 1 else {0: "13GiB"}

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()
model.config.use_cache = True
print("✅ Trained Socratic Model ready for testing!")

# Test Prompt
system_prompt = (
    "You are a Socratic Science Tutor for a Grade 10 student. "
    "Your goal is to guide the student to discover concepts through reasoning, NEVER by giving the final answer directly. "
    "Ask EXACTLY ONE question per turn. Keep responses to 1-3 sentences. "
    "If the student is stuck, provide a simpler analogy or break the concept into a smaller step. "
    "State your pedagogical goal inside <plan>...</plan> tags."
)

test_question = "Why do stars twinkle in the night sky, but planets don't?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_question}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_len = inputs["input_ids"].shape[1]
generated_response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print("\n--- Generated Socratic Response ---")
print(generated_response.strip())

## 8. Merge LoRA into 16-bit Standalone Model & Push to Hugging Face

In [ ]:
import os
import gc
import glob
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

OUTPUT_DIR = "./socratic_tutor_lora"
HUB_MODEL_ID = "Susu11/socratic_qwen8b"
MERGED_DIR = "./socratic_tutor_merged"
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def get_best_adapter_dir(output_dir="./socratic_tutor_lora"):
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        return output_dir
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
        return checkpoints[-1]
    return output_dir

adapter_dir = get_best_adapter_dir(OUTPUT_DIR)
print(f"📂 Using trained adapter from: {adapter_dir}")

# 1. Save Adapter to HUB if trainer is still in memory
if "trainer" in globals() and trainer is not None and hasattr(trainer, "model"):
    trainer.model.save_pretrained(OUTPUT_DIR)
    if "tokenizer" in globals():
        tokenizer.save_pretrained(OUTPUT_DIR)
    if os.getenv("HF_TOKEN"):
        try:
            trainer.model.push_to_hub(HUB_MODEL_ID)
            tokenizer.push_to_hub(HUB_MODEL_ID)
            print(f"🎉 LoRA adapter pushed to: https://huggingface.co/{HUB_MODEL_ID}")
        except Exception as e:
            print(f"ℹ️ Hub push notice: {e}")

# 2. Free VRAM before loading 16-bit base model
for var in ["model", "trainer", "base_model", "merged_model"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# 3. Merge Adapter into Base Model in 16-bit
print("⏳ Reloading base model in FP16 to merge LoRA weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"💾 Standalone 16-bit model saved locally to: {MERGED_DIR}")

# 4. Push Merged 16-bit Model to Hub
if os.getenv("HF_TOKEN"):
    merged_hub_id = f"{HUB_MODEL_ID}-merged"
    print(f"📤 Pushing full merged 16-bit model to: https://huggingface.co/{merged_hub_id}...")
    try:
        merged_model.push_to_hub(merged_hub_id)
        tokenizer.push_to_hub(merged_hub_id)
        print(f"🎉 Merged model successfully uploaded to: https://huggingface.co/{merged_hub_id}")
    except Exception as e:
        print(f"⚠️ Error pushing merged model: {e}")

## 9. 📦 Convert to GGUF Format (for Ollama, LM Studio, llama.cpp) & Upload to Hub
Converts your trained model to a `.gguf` file and uploads it to [`Susu11/socratic_qwen8b`](https://huggingface.co/Susu11/socratic_qwen8b).

In [ ]:
import os
import gc
import sys
import glob
import subprocess
import torch

# 1. Neutralize torchao dispatcher in PEFT
import peft.tuners.lora.torchao
peft.tuners.lora.torchao.dispatch_torchao = lambda *args, **kwargs: None

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import HfApi

OUTPUT_DIR = "./socratic_tutor_lora"
HUB_MODEL_ID = "Susu11/socratic_qwen8b"
MERGED_DIR = "./socratic_tutor_merged"
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUT_GGUF_NAME = "socratic_qwen_f16.gguf"

# 2. Locate trained adapter
def get_best_adapter_dir(output_dir="./socratic_tutor_lora"):
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        return output_dir
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
        return checkpoints[-1]
    return output_dir

adapter_dir = get_best_adapter_dir(OUTPUT_DIR)
print(f"📂 Using trained adapter from: {adapter_dir}")

# 3. Clean VRAM before merge
for var in ["base", "model", "trainer", "base_model", "merged_model"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# 4. Reload base model in FP16 across both GPUs & Merge
print("⏳ [Step 1/3] Reloading base model & merging LoRA weights (takes ~1 min)... ")
num_gpus = torch.cuda.device_count()
max_memory = {i: "13GiB" for i in range(num_gpus)}

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✅ Standalone 16-bit model successfully created at: {MERGED_DIR}")

# 5. Push Merged Model to Hub
if os.getenv("HF_TOKEN"):
    merged_hub_id = f"{HUB_MODEL_ID}-merged"
    print(f"\n📤 [Step 2/3] Uploading merged 16-bit model to: https://huggingface.co/{merged_hub_id}...")
    try:
        merged_model.push_to_hub(merged_hub_id)
        tokenizer.push_to_hub(merged_hub_id)
        print(f"🎉 Merged model live at: https://huggingface.co/{merged_hub_id}")
    except Exception as e:
        print(f"ℹ️ Hub push notice: {e}")

# Free memory before GGUF conversion
del base_model, merged_model
gc.collect()
torch.cuda.empty_cache()

# 6. Convert to GGUF and Upload
print(f"\n📦 [Step 3/3] Converting '{MERGED_DIR}' to '{OUT_GGUF_NAME}'...")
try:
    convert_cmd = [
        sys.executable,
        "llama.cpp/convert_hf_to_gguf.py",
        MERGED_DIR,
        "--outfile", OUT_GGUF_NAME,
        "--outtype", "f16"
    ]
    subprocess.run(convert_cmd, check=True)
    print(f"✅ GGUF created successfully: {OUT_GGUF_NAME} ({os.path.getsize(OUT_GGUF_NAME) / 1e9:.2f} GB)")

    if os.getenv("HF_TOKEN"):
        print(f"📤 Uploading GGUF to Hugging Face Hub: {HUB_MODEL_ID}...")
        api = HfApi()
        api.upload_file(
            path_or_fileobj=OUT_GGUF_NAME,
            path_in_repo=OUT_GGUF_NAME,
            repo_id=HUB_MODEL_ID,
            repo_type="model",
        )
        print(f"🎉 GGUF file uploaded successfully to: https://huggingface.co/{HUB_MODEL_ID}")
except Exception as e:
    print(f"⚠️ GGUF conversion/upload error: {e}")

print("\n🚀 ALL STEPS COMPLETED SUCCESSFULLY!")

In [ ]:
import os
import sys
import subprocess
from huggingface_hub import HfApi, snapshot_download

# Automatically retrieve HF token from Kaggle Secrets or environment
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    for key in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HF"]:
        try:
            hf_token = UserSecretsClient().get_secret(key)
            if hf_token: break
        except Exception:
            pass
except Exception:
    pass

if not hf_token:
    hf_token = os.getenv("HF_TOKEN")

assert hf_token, "⚠️ HF_TOKEN not found! Please attach your HF_TOKEN in Kaggle: Add-ons -> Secrets."

HUB_MODEL_ID = "Susu11/socratic_qwen8b"
MERGED_HUB_ID = f"{HUB_MODEL_ID}-merged"
LOCAL_MERGED_DIR = "./socratic_tutor_merged"
OUT_GGUF_NAME = "socratic_qwen_f16.gguf"

# 1. Locate model: Use local folder if available, otherwise fetch from Hugging Face Hub
if os.path.exists(LOCAL_MERGED_DIR) and os.path.exists(os.path.join(LOCAL_MERGED_DIR, "config.json")):
    model_dir = LOCAL_MERGED_DIR
    print(f"📂 Using local merged model at: {model_dir}")
else:
    print(f"🌐 Fetching your merged 16-bit model from Hub: {MERGED_HUB_ID}...")
    model_dir = snapshot_download(repo_id=MERGED_HUB_ID, token=hf_token)
    print(f"✅ Model downloaded to: {model_dir}")

# 2. Install gguf dependency
print("📦 Installing gguf library...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=True)

# 3. Ensure llama.cpp is present
if not os.path.exists("llama.cpp"):
    print("Cloning llama.cpp...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git"], check=True)

# 4. Convert model to GGUF
print(f"🔄 Converting to '{OUT_GGUF_NAME}' (takes ~2 mins)...")
convert_cmd = [
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    model_dir,
    "--outfile", OUT_GGUF_NAME,
    "--outtype", "f16"
]
subprocess.run(convert_cmd, check=True)
size_gb = os.path.getsize(OUT_GGUF_NAME) / 1e9
print(f"✅ GGUF created successfully: {OUT_GGUF_NAME} ({size_gb:.2f} GB)")

# 5. Upload GGUF directly to Hugging Face Hub
print(f"📤 Uploading GGUF to Hugging Face: https://huggingface.co/{HUB_MODEL_ID}...")
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=OUT_GGUF_NAME,
    path_in_repo=OUT_GGUF_NAME,
    repo_id=HUB_MODEL_ID,
    repo_type="model",
    token=hf_token,
)

print(f"\n🎉 GGUF file uploaded successfully!")
print(f"👉 View your GGUF model here: https://huggingface.co/{HUB_MODEL_ID}")

In [29]:
import os
import sys
import types

# 1. Disable wandb in environment
os.environ["WANDB_DISABLED"] = "true"

# 2. Inject a dummy mock for wandb so Python never touches broken disk files
if "wandb" not in sys.modules or not hasattr(sys.modules["wandb"], "errors"):
  dummy_wandb = types.ModuleType("wandb")
  dummy_wandb.errors = types.ModuleType("errors")
  dummy_wandb.errors.Error = Exception
  dummy_wandb.sdk = types.ModuleType("sdk")
  sys.modules["wandb"] = dummy_wandb
  sys.modules["wandb.errors"] = dummy_wandb.errors

# 3. Disable in transformers & trl
try:
  import transformers.integrations

  transformers.integrations.is_wandb_available = lambda: False
except Exception:
  pass
try:
  import transformers.integrations.integration_utils

  transformers.integrations.integration_utils.is_wandb_available = (
      lambda: False
  )
except Exception:
  pass

# 4. Now import DPO cleanly!
from trl import DPOConfig, DPOTrainer

print("🎉 DPOConfig and DPOTrainer imported successfully!")

RuntimeError: Failed to import trl.trainer.dpo_trainer because of the following error (look up to see its traceback):
wandb.__spec__ is None

In [ ]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    output_dir="./socratic_qwen_dpo",
    learning_rate=5e-7,  # Very gentle learning rate
    beta=0.1,  # Controls how strictly to penalize rejected answers
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
)

trainer = DPOTrainer(
    model=model,  # Your fine-tuned Qwen 7B model
    args=training_args,
    train_dataset=dpo_dataset,  # Dataset with prompt, chosen, rejected
    tokenizer=tokenizer,
)
trainer.train()